# AI-Based Food Inflation Prediction and Household Affordability Monitoring System
## Notebook 01: Data Collection & Corpus Overview

**Author:** Senior Data Engineer & Acquisition Specialist  
**Objective:** Inspect, validate, and explore the standardized multi-source datasets collected for the food inflation prediction and affordability monitoring system.

### Core Datasets Standardized & Verified:
1. **Retail Prices** (`data/processed/retail_prices.csv`) — Consumer retail quotes across Maharashtra centres (Mumbai, Pune, Nashik, Nagpur) and All-India benchmarks.
2. **Wholesale Mandi Prices** (`data/processed/wholesale_prices.csv`) — APMC mandi auction quotes (Min, Max, Modal prices in Rs/Quintal).
3. **Market Arrivals** (`data/processed/market_arrivals.csv`) — Physical supply volumes (Metric Tonnes) unloaded at APMC mandis.
4. **Consumer Price Index (CPI)** (`data/processed/cpi_data.csv`) — Official MoSPI monthly indices & YoY inflation rates (2014–2026).
5. **Household Expenditure (HCES)** (`data/processed/household_expenditure.csv`) — NSSO 2022-23 fractile MPCE & food group budget shares.

> **Methodological Rule:** Retail and wholesale prices are stored separately with distinct physical units (`Rs/Kg` vs `Rs/Quintal`). They are never mixed without explicit economic markup modeling.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Display configurations
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)

DATA_DIR = Path('../data/processed')
if not DATA_DIR.exists():
    DATA_DIR = Path('data/processed')

print(f"Loading processed datasets from: {DATA_DIR.resolve()}")

--- 
## 1. Consumer Retail Prices (`retail_prices.csv`)

In [ ]:
retail_path = DATA_DIR / 'retail_prices.csv'
if retail_path.exists():
    df_retail = pd.read_csv(retail_path)
    print(f"Retail Dataset Shape: {df_retail.shape}")
    print(f"Date Range: {df_retail['observation_date'].min()} to {df_retail['observation_date'].max()}")
    print(f"Unique Commodities: {df_retail['commodity_name'].nunique()}")
    print(f"Reporting Centres: {df_retail['reporting_centre'].unique()}")
    display(df_retail.head(5))
else:
    print("retail_prices.csv not yet generated. Run `py scripts/main.py --clean`.")

--- 
## 2. Wholesale Mandi Prices (`wholesale_prices.csv`)

In [ ]:
ws_path = DATA_DIR / 'wholesale_prices.csv'
if ws_path.exists():
    df_ws = pd.read_csv(ws_path)
    print(f"Wholesale Mandi Dataset Shape: {df_ws.shape}")
    print(f"Date Range: {df_ws['observation_date'].min()} to {df_ws['observation_date'].max()}")
    print(f"Unique Mandi Markets: {df_ws['market_name'].nunique()}")
    print(f"Districts Covered: {df_ws['district'].unique()[:10]}")
    print("\nModal Price (Rs/Qtl) Distribution by Commodity:")
    display(df_ws.groupby('commodity_name')['modal_price_rs_per_qtl'].describe().round(2))
    print(f"\nAnomalous Rows Flagged (min > max or modal outside bounds): {df_ws['is_anomalous_flag'].sum():,}")
    display(df_ws.head(5))
else:
    print("wholesale_prices.csv not yet generated. Run `py scripts/main.py --clean`.")

--- 
## 3. Physical Mandi Arrivals (`market_arrivals.csv`)

In [ ]:
arr_path = DATA_DIR / 'market_arrivals.csv'
if arr_path.exists():
    df_arr = pd.read_csv(arr_path)
    print(f"Market Arrivals Dataset Shape: {df_arr.shape}")
    print(f"Total Physical Volume Logged: {df_arr['arrival_quantity_tonnes'].sum():,.2f} Tonnes")
    display(df_arr.groupby('commodity_name')['arrival_quantity_tonnes'].agg(['count', 'mean', 'max', 'sum']).round(2))
    display(df_arr.head(5))
else:
    print("market_arrivals.csv not yet generated. Run `py scripts/main.py --clean`.")

--- 
## 4. Consumer Price Index & Inflation (`cpi_data.csv`)

In [ ]:
cpi_path = DATA_DIR / 'cpi_data.csv'
if cpi_path.exists():
    df_cpi = pd.read_csv(cpi_path)
    print(f"CPI Dataset Shape: {df_cpi.shape}")
    print(f"Temporal Span: {df_cpi['observation_date'].min()} to {df_cpi['observation_date'].max()}")
    print(f"Geographies Covered: {df_cpi['geography'].unique()}")
    print(f"Sectors: {df_cpi['sector'].unique()}")
    print("\nRecent Food Subgroup Inflation Rates (YoY %):")
    latest_date = df_cpi['observation_date'].max()
    display(df_cpi[(df_cpi['observation_date'] == latest_date) & (df_cpi['sector'] == 'Combined')][
        ['observation_date', 'geography', 'category_name', 'cpi_index', 'inflation_rate_yoy']
    ].head(10))
else:
    print("cpi_data.csv not yet generated. Run `py scripts/main.py --clean`.")

--- 
## 5. Household Consumption Expenditure Survey (`household_expenditure.csv`)

In [ ]:
hces_path = DATA_DIR / 'household_expenditure.csv'
if hces_path.exists():
    df_hces = pd.read_csv(hces_path)
    print(f"HCES Dataset Shape: {df_hces.shape}")
    print(f"Indicators Available:\n{df_hces['indicator_name'].value_counts()}")
    print("\nSample MPCE Across Fractile Classes (Bottom 5% to Top 5%):\n")
    display(df_hces[df_hces['indicator_name'] == 'MPCE across Fractile Classes'][
        ['sector', 'fractile_class', 'mpce_value_rs']
    ].head(12))
else:
    print("household_expenditure.csv not yet generated. Run `py scripts/main.py --clean`.")

--- 
## 6. Analytical Readiness Summary

| Pipeline Stage | Target Models / Methods | Data Sources Utilized | Status |
|---|---|---|:---:|
| **Mandi Price Forecasting** | SARIMA, Prophet, XGBoost, LSTM | `wholesale_prices.csv`, `market_arrivals.csv` | **Ready** |
| **Price Spike Anomaly Detection** | Isolation Forest, Rolling Z-score, DBSCAN | `wholesale_prices.csv` (`modal_price_rs_per_qtl`, `is_anomalous_flag`) | **Ready** |
| **Retail-Wholesale Spread Transmission** | Vector Autoregression (VAR), ECM | `retail_prices.csv` matched to `wholesale_prices.csv` | **Ready** |
| **Household Affordability Indexing** | Cost of Food Basket / Fractile MPCE | `retail_prices.csv`, `household_expenditure.csv` | **Ready** |
| **Macro Food Inflation Tracking** | CPI Group/Subgroup Forecasting | `cpi_data.csv` | **Ready** |